# Causal Conv1D Forward Operation

This notebook shows how to compute a causal depthwise 1D convolution forward operation using cuDNN.

$$y = \text{Activation}(\text{CausalConv1D}(x, w) + b)$$

where Activation is Identity or SiLU.

Where the convolution uses left-only (causal) padding of size `kernel_size - 1`, and each channel is convolved independently (depthwise).

## Prerequisites and Setup
This notebook requires an NVIDIA GPU (Hopper or later recommended).

**Environment setup** — make sure the cuDNN runtime library and the `cudnn` Python package are discoverable before launching the notebook:

- **Option A – pip install:**
  ```bash
  pip install nvidia-cudnn-frontend
  ```
- **Option B – set paths manually:**
  ```bash
  export LD_LIBRARY_PATH=/path/to/cudnn/lib:${LD_LIBRARY_PATH}
  export PYTHONPATH=/path/to/cudnn_frontend/build_python:${PYTHONPATH}
  ```

Adjust the paths above to match your local build or installation directory.

In [1]:
# !nvidia-smi

In [2]:
# get_ipython().system('pip install nvidia-cudnn-cu12')
# get_ipython().system('pip install nvidia-cudnn-frontend')
# get_ipython().system('pip3 install --pre torch --index-url https://download.pytorch.org/whl/nightly/cu128')

## Overview

We will perform a causal depthwise conv1d forward pass with:

- batch size: 2
- dim (channels): 768
- sequence length: 4096
- kernel size: 4
- activation: SiLU
- data type: BFloat16

We compare the cuDNN result against a PyTorch reference implementation.

In [3]:
import torch
import torch.nn.functional as F
import cudnn

print("cuDNN backend version:", cudnn.backend_version())

cuDNN backend version: 92200


## Reference Implementation

A simple PyTorch reference for causal depthwise conv1d + SiLU.

In [4]:
def causal_conv1d_ref(x, weight, bias, activation="silu"):
    """Reference: causal depthwise conv1d using PyTorch ops.

    Args:
        x:      (batch, dim, seq_len)
        weight: (dim, kernel_size)
        bias:   (dim,)
        activation: 'silu' or 'identity'

    Returns:
        y: (batch, dim, seq_len)
    """
    batch, dim, seq_len = x.shape
    kernel_size = weight.shape[1]

    # Causal padding: (kernel_size - 1) on the left, 0 on the right
    x_padded = F.pad(x, (kernel_size - 1, 0))

    # Depthwise conv1d: groups=dim
    # weight needs shape (dim, 1, kernel_size) for groups=dim
    w = weight.unsqueeze(1)  # (dim, 1, kernel_size)
    y = F.conv1d(x_padded, w, bias=bias, groups=dim)

    if activation == "silu":
        y = F.silu(y)

    return y

## Setup Tensors

In [5]:
batch = 2
dim = 768
seq_len = 4096
kernel_size = 4
dtype = torch.bfloat16

has_cuda = torch.cuda.is_available()

torch.manual_seed(42)

x = torch.randn(batch, dim, seq_len, dtype=dtype)
weight = torch.randn(dim, kernel_size, dtype=dtype)
bias = torch.randn(dim, dtype=dtype)

print(f"CUDA available: {has_cuda}")
print(f"x:      {x.shape}, dtype={x.dtype}")
print(f"weight: {weight.shape}, dtype={weight.dtype}")
print(f"bias:   {bias.shape}, dtype={bias.dtype}")

CUDA available: True
x:      torch.Size([2, 768, 4096]), dtype=torch.bfloat16
weight: torch.Size([768, 4]), dtype=torch.bfloat16
bias:   torch.Size([768]), dtype=torch.bfloat16


## Compute Reference Output

In [6]:
y_ref = causal_conv1d_ref(x, weight, bias, activation="silu")
print(f"y_ref:  {y_ref.shape}, dtype={y_ref.dtype}")

y_ref:  torch.Size([2, 768, 4096]), dtype=torch.bfloat16


## Compute cuDNN Output

In [7]:
if has_cuda:
    x_gpu = x.cuda()
    weight_gpu = weight.cuda()
    bias_gpu = bias.cuda()
    y_cudnn = cudnn.ops.causal_conv1d(x_gpu, weight_gpu, bias_gpu, activation="silu")
    print(f"y_cudnn: {y_cudnn.shape}, dtype={y_cudnn.dtype}")
else:
    print("Skipping cuDNN forward (no CUDA device). Run on a GPU machine to test cuDNN.")

y_cudnn: torch.Size([2, 768, 4096]), dtype=torch.bfloat16


## Verify Correctness

In [8]:
if has_cuda:
    y_cudnn_cpu = y_cudnn.cpu()
    max_abs_diff = (y_cudnn_cpu.float() - y_ref.float()).abs().max().item()
    max_rel_diff = ((y_cudnn_cpu.float() - y_ref.float()).abs() / (y_ref.float().abs() + 1e-6)).max().item()

    print(f"Max absolute difference: {max_abs_diff:.6e}")
    print(f"Max relative difference: {max_rel_diff:.6e}")

    atol = 1e-2
    rtol = 1e-2

    assert torch.allclose(y_cudnn_cpu.float(), y_ref.float(), atol=atol, rtol=rtol), \
        f"FAILED: max_abs={max_abs_diff}, max_rel={max_rel_diff}"

    print("PASSED: cuDNN causal_conv1d forward matches reference.")
else:
    print("Skipping verification (no CUDA device).")

Max absolute difference: 6.250000e-02
Max relative difference: 3.278166e-02
PASSED: cuDNN causal_conv1d forward matches reference.


## Test with Different Data Types and Kernel Sizes

In [9]:
tolerances = {
    torch.float32:  (5e-6, 5e-6),
    torch.float16:  (5e-3, 5e-3),
    torch.bfloat16: (1e-2, 1e-2),
}

test_configs = [
    {"dtype": torch.float32,  "kernel_size": 4,  "activation": "silu"},
    {"dtype": torch.float16,  "kernel_size": 4,  "activation": "silu"},
    {"dtype": torch.bfloat16, "kernel_size": 4,  "activation": "silu"},
    {"dtype": torch.bfloat16, "kernel_size": 3,  "activation": "silu"},
    {"dtype": torch.float16,  "kernel_size": 7,  "activation": "silu"},
    {"dtype": torch.bfloat16, "kernel_size": 4,  "activation": "identity"},
]

batch, dim, seq_len = 2, 256, 1024

for cfg in test_configs:
    dt = cfg["dtype"]
    ks = cfg["kernel_size"]
    act = cfg["activation"]
    atol, rtol = tolerances[dt]

    x = torch.randn(batch, dim, seq_len, dtype=dt)
    w = torch.randn(dim, ks, dtype=dt)
    b = torch.randn(dim, dtype=dt)

    y_ref = causal_conv1d_ref(x, w, b, activation=act)

    if has_cuda:
        y_cudnn = cudnn.ops.causal_conv1d(x.cuda(), w.cuda(), b.cuda(), activation=act)
        max_abs = (y_cudnn.cpu().float() - y_ref.float()).abs().max().item()
        passed = torch.allclose(y_cudnn.cpu().float(), y_ref.float(), atol=atol, rtol=rtol)
        status = "PASS" if passed else "FAIL"
        print(f"[{status}] dtype={str(dt):15s} K={ks:3d} act={act:10s} max_abs_diff={max_abs:.4e}")
        assert passed, f"Test failed for {cfg}"
    else:
        print(f"[REF ] dtype={str(dt):15s} K={ks:3d} act={act:10s} y_ref shape={y_ref.shape}")

print("\nAll tests passed!" if has_cuda else "\nAll reference tests completed (CPU only).")

[PASS] dtype=torch.float32   K=  4 act=silu       max_abs_diff=1.9073e-06
[PASS] dtype=torch.float16   K=  4 act=silu       max_abs_diff=7.8125e-03
[PASS] dtype=torch.bfloat16  K=  4 act=silu       max_abs_diff=6.2500e-02
[PASS] dtype=torch.bfloat16  K=  3 act=silu       max_abs_diff=6.2500e-02
[PASS] dtype=torch.float16   K=  7 act=silu       max_abs_diff=7.8125e-03
[PASS] dtype=torch.bfloat16  K=  4 act=identity   max_abs_diff=0.0000e+00

All tests passed!


## torch.compile Support

`cudnn.ops.causal_conv1d` is registered as a PyTorch custom operator, so it works seamlessly with `torch.compile`.

In [10]:
if has_cuda:
    B, D, L, K = 2, 256, 1024, 4
    x_c = torch.randn(B, D, L, dtype=torch.bfloat16, device="cuda")
    w_c = torch.randn(D, K, dtype=torch.bfloat16, device="cuda")
    b_c = torch.randn(D, dtype=torch.bfloat16, device="cuda")

    @torch.compile
    def compiled_causal_conv1d(x, w, b):
        return cudnn.ops.causal_conv1d(x, w, b, activation="silu")

    y_eager = cudnn.ops.causal_conv1d(x_c, w_c, b_c, activation="silu")
    y_compiled = compiled_causal_conv1d(x_c, w_c, b_c)

    max_abs = (y_compiled.float() - y_eager.float()).abs().max().item()
    print(f"torch.compile matches eager: max_abs_diff={max_abs:.4e}")
    assert max_abs == 0.0, f"torch.compile output differs from eager: {max_abs}"
    print(f"y_compiled: {y_compiled.shape}, dtype={y_compiled.dtype}")
    print("PASSED: torch.compile produces identical output to eager mode.")
else:
    print("Skipping torch.compile test (no CUDA device).")

torch.compile matches eager: max_abs_diff=0.0000e+00
y_compiled: torch.Size([2, 256, 1024]), dtype=torch.bfloat16
PASSED: torch.compile produces identical output to eager mode.
